# 03 Train Video Model

Train a CNN-LSTM hybrid model for deepfake detection. 
- **CNN Backbone**: EfficientNet-B0 (pretrained feature extractor).
- **LSTM**: To capture temporal dependencies between frames.
- **Loss**: Binary Cross Entropy with Logits.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import os
import cv2
from PIL import Image

# Hyperparameters
IMG_SIZE = 224
NUM_FRAMES = 16
BATCH_SIZE = 16
LR = 1e-4
EPOCHS = 10
HIDDEN_DIM = 256

DATA_ROOT = Path("../../data/splits/videos")
MODEL_DIR = Path("../../models/video")
MODEL_PATH = MODEL_DIR / "best_weights.pth"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Dataset & Data Loaders

In [ ]:
class VideoFrameDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, num_frames=16, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.num_frames = num_frames
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def _get_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            return None
        
        indices = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                if len(frames) > 0: frames.append(frames[-1])
                else: frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
                continue
            
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = Image.fromarray(frame)
            if self.transform:
                frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        return torch.stack(frames)
    
    def __getitem__(self, index):
        row = self.df.iloc[index]
        frames = self._get_frames(row["path"])
        if frames is None:
            frames = torch.zeros((self.num_frames, 3, IMG_SIZE, IMG_SIZE))
        return frames, torch.tensor(row["label_id"], dtype=torch.float32)

def collect_data(root_path):
    data = []
    for split in ["train", "val"]:
        for label in ["real", "fake"]:
            folder = root_path / split / label
            for video_path in folder.glob("*.mp4"):
                data.append({"path": str(video_path), "split": split, "label_id": 1 if label == "fake" else 0})
    return pd.DataFrame(data)

df_all = collect_data(DATA_ROOT)
train_df = df_all[df_all["split"] == "train"]
val_df = df_all[df_all["split"] == "val"]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(VideoFrameDataset(train_df, num_frames=NUM_FRAMES, transform=transform), 
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(VideoFrameDataset(val_df, num_frames=NUM_FRAMES, transform=transform), 
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 99 | Val batches: 22


## 2. Model Architecture (CNN-LSTM)

In [3]:
class DeepfakeVideoModel(nn.Module):
    def __init__(self, hidden_dim=256):
        super(DeepfakeVideoModel, self).__init__()
        # CNN Backbone (EfficientNet-B0)
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1]) # Remove classifier
        
        # Freeze backbone (optional: keep True for initial transfer learning)
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
        
        self.lstm = nn.LSTM(input_size=1280, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        # x shape: (Batch, Time, Channels, H, W)
        batch_size, time_steps, C, H, W = x.size()
        
        # Reshape for CNN: (Batch * Time, C, H, W)
        x = x.view(batch_size * time_steps, C, H, W)
        features = self.feature_extractor(x)
        features = features.view(batch_size * time_steps, -1)
        
        # Reshape for LSTM: (Batch, Time, Features)
        features = features.view(batch_size, time_steps, -1)
        
        lstm_out, (h_n, c_n) = self.lstm(features)
        
        # Use the last hidden state for classification
        last_hidden = h_n[-1]
        out = self.classifier(last_hidden)
        return out

model = DeepfakeVideoModel(hidden_dim=HIDDEN_DIM).to(device)
print(f"Model initialized and moved to {device}")

Model initialized and moved to cuda


## 3. Training Loop

In [4]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

best_val_auc = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for frames, labels in pbar:
        frames, labels = frames.to(device), labels.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    # Validation
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for frames, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            frames = frames.to(device)
            outputs = model(frames)
            probs = torch.sigmoid(outputs)
            
            val_preds.extend(probs.cpu().numpy())
            val_labels.extend(labels.numpy())
    
    val_auc = roc_auc_score(val_labels, val_preds)
    val_acc = accuracy_score(val_labels, np.array(val_preds) >= 0.5)
    
    print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f}")
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"--> Saved best model weights to {MODEL_PATH}")

Epoch 1/10 [Train]:   0%|          | 0/99 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/22 [00:00<?, ?it/s]

Epoch 1: Train Loss: 0.4408 | Val Acc: 0.8787 | Val AUC: 0.5609
--> Saved best model weights to ..\..\models\video\best_weights.pth


Epoch 2/10 [Train]:   0%|          | 0/99 [00:00<?, ?it/s]

Epoch 2/10 [Val]:   0%|          | 0/22 [00:00<?, ?it/s]

...

Epoch 9/10 [Val]:   0%|          | 0/22 [00:00<?, ?it/s]

Epoch 9: Train Loss: 0.2196 | Val Acc: 0.8580 | Val AUC: 0.7117


Epoch 10/10 [Train]:   0%|          | 0/99 [00:00<?, ?it/s]

Epoch 10/10 [Val]:   0%|          | 0/22 [00:00<?, ?it/s]

Epoch 10: Train Loss: 0.2124 | Val Acc: 0.8698 | Val AUC: 0.6972
